# 🩺 Diabetic Retinopathy Grading — Elite Pipeline v17
## 2-Stage + Ordinal | 5-Fold CV | Ensemble | TTA | Grad-CAM++
### ✅ Colab / Kaggle / Local GPU Compatible
---
**Run order:** Cell 1 (Install) → Cell 2 (Imports) → **Cell 2b (Download Dataset)** → Cell 3 onwards
---
**Key Improvements over v16:**
- Fixed threshold optimisation (proper scipy.optimize ordinal cutpoint search)
- Fixed AMP (mixed precision) in Stage 1 & 2 training loops
- Added Phase 3 (512 px) to K-Fold progressive resizing
- Fixed ensemble TTA count bug
- Added WeightedRandomSampler oversampling
- Added multi-backbone support (EfficientNetV2-B1 + ConvNeXt-Tiny + Swin-Tiny)
- Fixed `GaussNoise` deprecation in albumentations ≥ 1.4
- Colab + Windows + Kaggle path auto-detection


## 📦 Cell 1 — Install / Upgrade Dependencies

In [ ]:
import sys, subprocess

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

print("Installing dependencies ...")
pip("--upgrade",
    "timm>=1.0.3",
    "albumentations>=1.4.0,<2.0.0",
    "opencv-python-headless",
    "scikit-learn", "pandas", "numpy", "tqdm", "matplotlib",
    "grad-cam",
    "scipy",
    "pyarrow", "fastparquet",
    "gradio>=4.44.1",
    "huggingface_hub>=0.23.0",
)
print("✅ All dependencies installed.")


## ⚙️ Cell 2 — Imports, Device, Paths, Resume Infrastructure

In [ ]:
import os, sys, io, json, gc, time, random, shutil, warnings, pickle
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from albumentations import __version__ as A_VER

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, cohen_kappa_score, ConfusionMatrixDisplay
)
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# ── Seed ─────────────────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🔥 GPU: {gpu_name}  VRAM: {vram:.1f} GB  CUDA: {torch.version.cuda}")
else:
    DEVICE = "cpu"
    vram = 0
    print("💻 No CUDA GPU — running on CPU (slow)")

USE_AMP = (DEVICE == "cuda")

# ── Environment detection ─────────────────────────────────────────────────────
IN_COLAB   = "google.colab" in sys.modules
IN_KAGGLE  = os.path.exists("/kaggle/input")
IN_WINDOWS = sys.platform == "win32"

if IN_KAGGLE:
    DATA_DIR     = Path("/kaggle/input/aptos2019-blindness-detection")
    ARTIFACT_DIR = Path("/kaggle/working/artifacts_v17")
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DATA_DIR     = Path("/content/drive/MyDrive/DR_data/aptos2019")
    ARTIFACT_DIR = Path("/content/drive/MyDrive/DR_data/artifacts_v17")
else:  # local (Windows / Linux)
    DATA_DIR     = Path(os.environ.get("DATA_DIR",     str(Path.home()/"DR_data"/"aptos2019")))
    ARTIFACT_DIR = Path(os.environ.get("ARTIFACT_DIR", str(Path.home()/"DR_data"/"artifacts_v17")))

IMG_DIR  = DATA_DIR / "train_images"
CSV_PATH = DATA_DIR / "train.csv"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = ARTIFACT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ────────────────────────────────────────────────────────────────
NUM_CLASSES  = 5
N_FOLDS      = 5
GRADE_MAP    = {0:"No DR",1:"Mild",2:"Moderate",3:"Severe",4:"Proliferative"}
GRADE_COLORS = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]

# ── State helpers (resume across cells) ──────────────────────────────────────
_STATE_FILE = ARTIFACT_DIR / "state_v17.json"
def st_load():
    if _STATE_FILE.exists():
        return json.loads(_STATE_FILE.read_text())
    return {}
def st_save(key, val):
    s = st_load(); s[key] = val
    _STATE_FILE.write_text(json.dumps(s, indent=2))

# ── PyTorch safe-load ────────────────────────────────────────────────────────
def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except Exception:
        return torch.load(path, map_location=map_location)

print(f"\n✅ PyTorch {torch.__version__} | timm {timm.__version__} | albumentations {A_VER}")
print(f"   Device:{DEVICE.upper()}  AMP:{'ON' if USE_AMP else 'OFF'}")
print(f"   DATA_DIR: {DATA_DIR}")
print(f"   ARTIFACT_DIR: {ARTIFACT_DIR}")


## 📥 Cell 2b — Dataset Download (APTOS 2019)
Automatically downloads **aptos2019-blindness-detection** from Kaggle.

| Environment | Method |
|-------------|--------|
| **Kaggle Notebook** | Data already mounted at `/kaggle/input/` — skipped automatically |
| **Google Colab** | Upload `kaggle.json` when prompted, then downloads & unzips |
| **Windows / Linux local** | Place `kaggle.json` in `~/.kaggle/` (or set `KAGGLE_USERNAME` + `KAGGLE_KEY` env vars) |

> **Get your kaggle.json**: kaggle.com → Account → API → "Create New Token"


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2b — APTOS 2019 Dataset Download                           ║
# ║  Safe to re-run: skips download if data already present          ║
# ╚══════════════════════════════════════════════════════════════════╝

import os, sys, zipfile, shutil, subprocess
from pathlib import Path

# ── These were already defined in Cell 2 ────────────────────────────
# DATA_DIR, IMG_DIR, CSV_PATH, IN_COLAB, IN_KAGGLE
# If running this cell standalone, define them here:
try:
    _ = DATA_DIR
except NameError:
    IN_COLAB  = "google.colab" in sys.modules
    IN_KAGGLE = os.path.exists("/kaggle/input")
    if IN_KAGGLE:
        DATA_DIR = Path("/kaggle/input/aptos2019-blindness-detection")
    elif IN_COLAB:
        DATA_DIR = Path("/content/drive/MyDrive/DR_data/aptos2019")
    else:
        DATA_DIR = Path.home() / "DR_data" / "aptos2019"
    IMG_DIR  = DATA_DIR / "train_images"
    CSV_PATH = DATA_DIR / "train.csv"

COMPETITION = "aptos2019-blindness-detection"

# ── Helper ───────────────────────────────────────────────────────────
def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

def _check_complete():
    """Return True if train.csv and at least 3000 images are present."""
    if not CSV_PATH.exists():
        return False
    n_imgs = len(list(IMG_DIR.glob("*.png"))) if IMG_DIR.exists() else 0
    return n_imgs >= 3000

# ── Already downloaded? ───────────────────────────────────────────────
if _check_complete():
    n = len(list(IMG_DIR.glob("*.png")))
    print(f"✅ Dataset already present: {CSV_PATH.parent}")
    print(f"   train.csv ✓  |  {n} train images ✓")

# ── Kaggle Notebook: data is auto-mounted, just symlink ───────────────
elif IN_KAGGLE:
    src = Path(f"/kaggle/input/{COMPETITION}")
    if src.exists():
        DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
        if not DATA_DIR.exists():
            DATA_DIR.symlink_to(src)
        print(f"✅ Kaggle: data symlinked from {src} → {DATA_DIR}")
    else:
        print(f"⚠️  In Kaggle but competition data not found at {src}")
        print(f"   Add '{COMPETITION}' as a Dataset in the right panel.")

# ── Google Colab ──────────────────────────────────────────────────────
elif IN_COLAB:
    _pip("kaggle")
    from google.colab import files as _colab_files

    kaggle_cfg = Path.home() / ".kaggle" / "kaggle.json"
    if not kaggle_cfg.exists():
        print("📤 Upload your kaggle.json (kaggle.com → Account → API → Create New Token)")
        _uploaded = _colab_files.upload()
        kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
        for fname, data in _uploaded.items():
            kaggle_cfg.write_bytes(data)
        kaggle_cfg.chmod(0o600)
        print(f"✅ kaggle.json saved → {kaggle_cfg}")

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _zip = DATA_DIR / f"{COMPETITION}.zip"

    if not _zip.exists():
        print(f"⬇️  Downloading {COMPETITION} (~1.5 GB) ...")
        subprocess.check_call([
            sys.executable, "-m", "kaggle", "competitions", "download",
            "-c", COMPETITION, "-p", str(DATA_DIR)
        ])
        print("✅ Download complete.")

    # Unzip
    print("📦 Unzipping ...")
    with zipfile.ZipFile(_zip, "r") as z:
        z.extractall(DATA_DIR)

    # Unzip nested zips (train_images.zip, test_images.zip)
    for nested in DATA_DIR.glob("*.zip"):
        print(f"   Unzipping nested: {nested.name}")
        with zipfile.ZipFile(nested, "r") as z:
            z.extractall(DATA_DIR)
        nested.unlink()

    _zip.unlink(missing_ok=True)
    n = len(list(IMG_DIR.glob("*.png")))
    print(f"✅ Extraction complete: {n} images in {IMG_DIR}")

# ── Local Windows / Linux ─────────────────────────────────────────────
else:
    _pip("kaggle")

    # Look for kaggle.json
    kaggle_cfg = Path.home() / ".kaggle" / "kaggle.json"
    env_key    = os.environ.get("KAGGLE_KEY", "")
    env_user   = os.environ.get("KAGGLE_USERNAME", "")

    if not kaggle_cfg.exists() and not (env_key and env_user):
        print("=" * 60)
        print("  ⚙️  KAGGLE API SETUP REQUIRED (one-time)")
        print("=" * 60)
        print("  1. Go to https://www.kaggle.com/settings/account")
        print("  2. Scroll to 'API' section → click 'Create New Token'")
        print("  3. A file 'kaggle.json' will be downloaded")
        print(f"  4. Move it to: {kaggle_cfg.parent}")
        print("     (Windows: C:\\Users\\<YourName>\\.kaggle\\kaggle.json)")
        print()
        print("  OR set environment variables:")
        print("     KAGGLE_USERNAME=<your_username>")
        print("     KAGGLE_KEY=<your_api_key>")
        print("=" * 60)
        raise EnvironmentError("kaggle.json not found. See instructions above.")

    # Ensure permissions
    if kaggle_cfg.exists():
        try: kaggle_cfg.chmod(0o600)
        except Exception: pass

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _zip = DATA_DIR / f"{COMPETITION}.zip"

    if not _zip.exists():
        print(f"⬇️  Downloading APTOS 2019 (~1.5 GB) to {DATA_DIR} ...")
        print("   This is a one-time download (~3–10 min depending on internet speed)")
        result = subprocess.run([
            sys.executable, "-m", "kaggle", "competitions", "download",
            "-c", COMPETITION, "-p", str(DATA_DIR)
        ], capture_output=True, text=True)
        if result.returncode != 0:
            print("STDERR:", result.stderr)
            raise RuntimeError(f"Kaggle download failed:\n{result.stderr}")
        print("✅ Download complete.")
    else:
        print(f"✅ ZIP already downloaded: {_zip}")

    # Unzip main archive
    print("📦 Unzipping main archive ...")
    with zipfile.ZipFile(_zip, "r") as z:
        members = z.namelist()
        print(f"   Contents: {members[:5]} {'...' if len(members)>5 else ''}")
        z.extractall(DATA_DIR)

    # Unzip any nested zips
    for nested in DATA_DIR.glob("*.zip"):
        print(f"📦 Unzipping nested: {nested.name}")
        with zipfile.ZipFile(nested, "r") as z:
            z.extractall(DATA_DIR)
        nested.unlink()

    _zip.unlink(missing_ok=True)
    n = len(list(IMG_DIR.glob("*.png")))
    print(f"✅ Done: {n:,} train images  |  CSV: {CSV_PATH.exists()}")

# ── Final verification ────────────────────────────────────────────────
print()
print("─" * 45)
print("DATASET VERIFICATION")
print("─" * 45)
for label, path in [("train.csv", CSV_PATH),
                     ("train_images/", IMG_DIR),
                     ("test_images/",  DATA_DIR/"test_images")]:
    status = "✅" if path.exists() else "❌ MISSING"
    extra  = ""
    if path.exists() and path.is_dir():
        extra = f"  ({len(list(path.glob('*.png')))} .png files)"
    print(f"  {status}  {label}{extra}")
print("─" * 45)

if not _check_complete():
    raise RuntimeError("Dataset incomplete after download. Check errors above.")
print("✅ Dataset ready for training!")


## 📥 Cell 3 — Load & Verify Dataset

In [ ]:
if not CSV_PATH.exists():
    print(f"❌ train.csv not found at {CSV_PATH}")
    print("   On Kaggle: Data is auto-mounted — check /kaggle/input/aptos2019-blindness-detection/")
    print("   Local: kaggle competitions download -c aptos2019-blindness-detection")
    raise FileNotFoundError(f"train.csv missing: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
df["image_path"]  = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
df["binary"]      = (df["diagnosis"] >= 1).astype(int)

missing = df[~df["image_path"].apply(lambda p: Path(p).exists())]
if len(missing):
    print(f"⚠️  {len(missing)} missing images — dropping them")
    df = df[df["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
else:
    print(f"✅ All {len(df)} images verified.")

print(f"\nClass distribution (imbalance ratio: {df['diagnosis'].value_counts().max()/df['diagnosis'].value_counts().min():.1f}x):")
for g in range(5):
    n   = (df["diagnosis"] == g).sum()
    pct = n / len(df) * 100
    bar = "█" * (n // 50)
    print(f"  Grade {g} ({GRADE_MAP[g]:15s}): {n:5d}  {pct:4.1f}%  {bar}")


## 🔬 Cell 4 — Fundus Preprocessing
Ben Graham method: CLAHE → Gaussian-blur subtraction → green channel boost

In [ ]:
IMG_SIZE      = int(os.environ.get("IMG_SIZE", 512))
BG_SIGMA      = max((IMG_SIZE // 10) | 1, 1)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def _make_circular_mask(img):
    h, w = img.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    cv2.circle(mask, (w//2, h//2), int(min(h, w)//2 * 0.97), 255, -1)
    return mask

def _clahe_lab(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def _green_emphasis(rgb):
    out = rgb.copy().astype(np.float32)
    out[:, :, 1] = np.clip(out[:, :, 1] * 1.1, 0, 255)
    return out.astype(np.uint8)

def preprocess_fundus(path_or_array, size=None):
    """
    Full fundus preprocessing:
    1. Load RGB
    2. Auto-crop black borders
    3. Resize + pad to square
    4. Circular mask
    5. CLAHE (LAB space)
    6. Ben Graham sharpening (4×img - 4×blur + 128)
    7. Green channel emphasis
    Returns: uint8 RGB [size × size × 3]
    """
    target = size or IMG_SIZE

    if isinstance(path_or_array, np.ndarray):
        rgb = path_or_array
    else:
        bgr = cv2.imread(str(path_or_array))
        if bgr is None:
            return np.zeros((target, target, 3), np.uint8)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Auto-crop black borders
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        rgb = rgb[y:y+h, x:x+w]

    # Resize keeping aspect ratio → pad to square
    h, w = rgb.shape[:2]
    scale = target / max(h, w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    rgb = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA)
    pt = (target - nh) // 2; pb = target - nh - pt
    pl = (target - nw) // 2; pr = target - nw - pl
    rgb = cv2.copyMakeBorder(rgb, pt, pb, pl, pr, cv2.BORDER_REFLECT_101)

    # Circular mask
    mask = _make_circular_mask(rgb)
    rgb[mask == 0] = 0

    # CLAHE
    rgb = _clahe_lab(rgb)

    # Ben Graham sharpening: 4×img − 4×blur + 128
    sig  = max((target // 10) | 1, 1)
    blur = cv2.GaussianBlur(rgb, (0, 0), sigmaX=sig)
    rgb  = cv2.addWeighted(rgb, 4, blur, -4, 128)
    rgb[mask == 0] = 0

    # Green channel emphasis
    rgb = _green_emphasis(rgb)
    return rgb

print(f"✅ preprocess_fundus defined  (IMG_SIZE={IMG_SIZE}, sigma={BG_SIGMA})")

# Latency check
if "df" in dir():
    t = time.time()
    for _ in range(3): preprocess_fundus(df["image_path"].iloc[0])
    lat = (time.time() - t) / 3 * 1000
    ok  = "✅" if lat < 80 else "⚠️ slow"
    print(f"   Preprocessing latency: {lat:.1f} ms / image  {ok}")


## 🔀 Cell 5 — Augmentation Pipelines (Albumentations)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from packaging.version import Version

_A_NEW = Version(A.__version__) >= Version("1.4.0")  # API changed in 1.4

def _gauss_noise():
    """GaussNoise API changed in albumentations 1.4 — handle both."""
    if _A_NEW:
        return A.GaussNoise(std_range=(0.03, 0.12), p=0.2)
    else:
        return A.GaussNoise(var_limit=(10.0, 50.0), p=0.2)

def _coarse_dropout(img_size):
    hole = img_size // 16
    if _A_NEW:
        return A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(hole, hole),
                               hole_width_range=(hole, hole), p=0.2)
    else:
        return A.CoarseDropout(max_holes=8, max_height=hole, max_width=hole,
                               min_holes=1, p=0.2)

def build_train_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.RandomResizedCrop(height=img_size, width=img_size,
                            scale=(0.7, 1.0), ratio=(0.9, 1.1), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=15, p=0.7),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20,
                             val_shift_limit=10, p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        _gauss_noise(),
        A.MotionBlur(blur_limit=3, p=0.1),
        _coarse_dropout(img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def build_val_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

# 5-view TTA
def build_tta_transforms(img_size=IMG_SIZE):
    norm = [A.Resize(img_size, img_size),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return [
        A.Compose(norm),                                       # original
        A.Compose([A.HorizontalFlip(p=1.0)] + norm),
        A.Compose([A.VerticalFlip(p=1.0)]   + norm),
        A.Compose([A.Rotate(limit=10, p=1.0)] + norm),
        A.Compose([A.Rotate(limit=-10, p=1.0)] + norm),
    ]

train_transforms = build_train_transforms(IMG_SIZE)
val_transforms   = build_val_transforms(IMG_SIZE)
tta_list         = build_tta_transforms(IMG_SIZE)

print(f"✅ Augmentation pipelines built (train: {len(train_transforms.transforms)} ops, TTA: {len(tta_list)} views)")


## 📚 Cell 6 — Dataset Classes

In [ ]:
class APTOSDataset(Dataset):
    """General-purpose dataset — returns (image_tensor, grade_label)."""

    def __init__(self, df, transform=None, preprocess=True, img_size=None):
        self.df         = df.reset_index(drop=True)
        self.transform  = transform
        self.preprocess = preprocess
        self.img_size   = img_size or IMG_SIZE

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row["diagnosis"])
        if self.preprocess:
            img = preprocess_fundus(row["image_path"], size=self.img_size)
        else:
            bgr = cv2.imread(str(row["image_path"]))
            img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.img_size, self.img_size))
        if self.transform:
            img = self.transform(image=img)["image"]
        else:
            img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
        return img, label


class BinaryAPTOSDataset(Dataset):
    """Stage-1 binary: 0 = No DR,  1 = any DR (grades 1-4)."""

    def __init__(self, df, transform=None, img_size=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.img_size  = img_size or IMG_SIZE

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = float(row["diagnosis"] >= 1)
        img   = preprocess_fundus(row["image_path"], size=self.img_size)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, torch.tensor(label, dtype=torch.float32)


class OrdinalAPTOSDataset(Dataset):
    """Stage-2 ordinal: DR-positive only. Grade → cumulative binary vector.
       1→[1,0,0,0]  2→[1,1,0,0]  3→[1,1,1,0]  4→[1,1,1,1]
    """
    ORDINAL = {1:[1,0,0,0], 2:[1,1,0,0], 3:[1,1,1,0], 4:[1,1,1,1]}

    def __init__(self, df, transform=None, img_size=None):
        self.df        = df[df["diagnosis"] >= 1].reset_index(drop=True)
        self.transform = transform
        self.img_size  = img_size or IMG_SIZE

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        grade = int(row["diagnosis"])
        label = torch.tensor(self.ORDINAL[grade], dtype=torch.float32)
        img   = preprocess_fundus(row["image_path"], size=self.img_size)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label


def make_weighted_loader(df_split, dataset, batch_size, drop_last=False):
    """DataLoader with WeightedRandomSampler for oversampling minority classes."""
    labels  = df_split["diagnosis"].values
    counts  = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    weights = 1.0 / np.maximum(counts, 1)
    sample_weights = torch.tensor([weights[l] for l in labels], dtype=torch.float)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights),
                                    replacement=True)
    nw = min(4, os.cpu_count() or 1)
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=nw, pin_memory=(DEVICE=="cuda"),
                      drop_last=drop_last, persistent_workers=(nw > 0))

def make_loader(dataset, batch_size, shuffle=False, drop_last=False):
    nw = min(4, os.cpu_count() or 1)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=nw, pin_memory=(DEVICE=="cuda"),
                      drop_last=drop_last, persistent_workers=(nw > 0))

print("✅ Dataset classes defined: APTOSDataset, BinaryAPTOSDataset, OrdinalAPTOSDataset")
print("   DataLoader helpers: make_weighted_loader (oversampling), make_loader")


## 🏗️ Cell 7 — Model Architecture
EfficientNetV2-B1 | ConvNeXt-Tiny | Swin-Tiny  +  GeM Pooling

In [ ]:
BACKBONE = os.environ.get("BACKBONE", "tf_efficientnetv2_b1")

# Supported backbones for ensemble
BACKBONE_REGISTRY = {
    "tf_efficientnetv2_b1": "tf_efficientnetv2_b1",
    "convnext_tiny":         "convnext_tiny",
    "swin_tiny_patch4_window7_224": "swin_tiny_patch4_window7_224",
}


class GeM(nn.Module):
    """Generalized Mean Pooling — learns optimal pooling exponent p."""
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1))
        ).pow(1.0 / self.p)


class DRModel(nn.Module):
    """
    Unified DR model for:
      - 5-class softmax  (num_classes=5, mode="softmax")
      - binary sigmoid   (num_classes=1, mode="sigmoid")
      - ordinal sigmoid  (num_classes=4, mode="sigmoid")
    """
    def __init__(self, backbone=BACKBONE, num_classes=5,
                 dropout=0.4, pretrained=True, mode="softmax"):
        super().__init__()
        self.mode = mode
        self.backbone = timm.create_model(
            backbone, pretrained=pretrained,
            features_only=False, num_classes=0, global_pool=""
        )
        feat_dim = self.backbone.num_features
        # Swin outputs (B, L, C) — need spatial reshape; use adaptive pooling instead
        self._is_transformer = "swin" in backbone.lower()
        self.pool = GeM(p=3)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        feat = self.backbone(x)
        if self._is_transformer:
            # Swin/ViT returns (B, H*W, C) or (B, C) depending on global_pool
            if feat.dim() == 3:            # (B, tokens, C) → (B, C, H, W) approx
                B, T, C = feat.shape
                H = W = int(T**0.5)
                feat = feat.permute(0, 2, 1).reshape(B, C, H, W)
            elif feat.dim() == 2:
                feat = feat.unsqueeze(-1).unsqueeze(-1)
        pooled = self.pool(feat)
        return self.head(pooled)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)

    def unfreeze_top_blocks(self, n=4):
        for p in self.backbone.parameters(): p.requires_grad_(False)
        if hasattr(self.backbone, "blocks"):
            for blk in list(self.backbone.blocks)[-n:]:
                for p in blk.parameters(): p.requires_grad_(True)
        for attr in ["conv_head", "bn2", "norm_head", "norm"]:
            if hasattr(self.backbone, attr):
                for p in getattr(self.backbone, attr).parameters(): p.requires_grad_(True)

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad_(True)


def build_model(num_classes=5, backbone=BACKBONE, pretrained=True,
                mode="softmax", dropout=0.4):
    return DRModel(backbone, num_classes, dropout, pretrained, mode).to(DEVICE)

def build_5class_model(pretrained=True, backbone=BACKBONE):
    return build_model(5, backbone, pretrained, "softmax")

def build_stage1_model(pretrained=True, backbone=BACKBONE):
    return build_model(1, backbone, pretrained, "sigmoid")

def build_stage2_model(pretrained=True, backbone=BACKBONE):
    return build_model(4, backbone, pretrained, "sigmoid")

# Sanity forward pass
_dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
_m     = build_5class_model(pretrained=False)
_out   = _m(_dummy)
print(f"✅ DRModel forward pass: input {list(_dummy.shape)} → output {list(_out.shape)}")
del _m, _dummy, _out; gc.collect()


## ⚖️ Cell 8 — Loss Functions, MixUp & Metrics

In [ ]:
class FocalLoss(nn.Module):
    """Multi-class Focal Loss with optional class weights."""
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.reduction = reduction

    def forward(self, inputs, targets):
        ce   = F.cross_entropy(inputs, targets, weight=self.alpha, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction == "mean" else loss.sum()


class BinaryFocalLoss(nn.Module):
    """Binary Focal Loss for Stage 1."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma

    def forward(self, logits, targets):
        bce    = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob   = torch.sigmoid(logits)
        p_t    = prob * targets + (1 - prob) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()


class OrdinalBCELoss(nn.Module):
    """Per-node BCE for ordinal stage-2 model."""
    def forward(self, logits, targets):
        return F.binary_cross_entropy_with_logits(logits, targets)


def compute_class_weights(labels, num_classes=5):
    counts  = np.bincount(labels, minlength=num_classes).astype(float)
    weights = len(labels) / (num_classes * np.maximum(counts, 1))
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)


def build_5class_criterion(df_split, device=DEVICE):
    cw  = compute_class_weights(df_split["diagnosis"].values).to(device)
    ce  = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    fl  = FocalLoss(alpha=cw, gamma=2.0)
    def criterion(logits, labels):
        return 0.5 * ce(logits, labels) + 0.5 * fl(logits, labels)
    return criterion


def mixup_data(x, y, alpha=0.4, device=DEVICE):
    lam = max(np.random.beta(alpha, alpha), 1 - np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0)).to(device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def qwk(y_true, y_pred):
    return cohen_kappa_score(np.array(y_true), np.array(y_pred), weights="quadratic")

def acc(y_true, y_pred):
    return (np.array(y_true) == np.array(y_pred)).mean()

print("✅ Loss functions: FocalLoss, BinaryFocalLoss, OrdinalBCELoss")
print("   Helpers: compute_class_weights, MixUp, qwk(), acc()")


## 📊 Cell 9 — EDA: Class Distribution

In [ ]:
_eda_flag = PLOT_DIR / "eda_distribution.png"

if _eda_flag.exists():
    print("✅ [RESUME] EDA already done — loading plot.")
    plt.figure(figsize=(10, 4))
    plt.imshow(plt.imread(str(_eda_flag)))
    plt.axis("off"); plt.tight_layout(); plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    counts = [df["diagnosis"].value_counts().sort_index().get(i, 0) for i in range(5)]
    labels = [f"G{i}\n{GRADE_MAP[i]}" for i in range(5)]

    axes[0].bar(labels, counts, color=GRADE_COLORS, edgecolor="black", linewidth=0.5)
    axes[0].set_title("Grade Distribution (Absolute)", fontsize=13, fontweight="bold")
    axes[0].set_ylabel("Count")
    for bar, cnt in zip(axes[0].patches, counts):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                     str(cnt), ha="center", va="bottom", fontsize=10)

    pcts = [c / sum(counts) * 100 for c in counts]
    axes[1].pie(pcts, labels=labels, colors=GRADE_COLORS, autopct="%1.1f%%",
                startangle=140, pctdistance=0.82)
    axes[1].set_title("Grade Distribution (%)", fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.savefig(str(_eda_flag), dpi=120, bbox_inches="tight")
    plt.show()
    print(f"✅ EDA plot saved → {_eda_flag}")


## ✂️ Cell 10 — Stratified 5-Fold Split

In [ ]:
_splits_path = ARTIFACT_DIR / "kfold_splits.parquet"

if _splits_path.exists():
    df = pd.read_parquet(_splits_path)
    print("✅ [RESUME] K-Fold splits loaded from disk.")
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    df["fold"] = -1
    for fold_idx, (_, val_idx) in enumerate(skf.split(df, df["diagnosis"])):
        df.loc[val_idx, "fold"] = fold_idx
    df.to_parquet(_splits_path, index=False)
    print("✅ 5-Fold splits created and saved.")

print("\nFold distribution:")
for fold in range(N_FOLDS):
    n  = (df["fold"] == fold).sum()
    gd = df[df["fold"] == fold]["diagnosis"].value_counts().sort_index()
    gs = " | ".join([f"G{g}:{c}" for g, c in gd.items()])
    print(f"  Fold {fold}: {n:5d} samples  [{gs}]")


## 🏋️ Cell 11 — Training Engine (Single-Epoch Loops)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer,
                    scaler=None, use_mixup=True, grad_accum=1, device=DEVICE):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    optimizer.zero_grad()

    for step, (imgs, labels) in enumerate(tqdm(loader, desc="  train", leave=False)):
        imgs   = imgs.to(device)
        labels = labels.to(device)

        y_a = y_b = labels  # defaults if no mixup
        lam = 1.0
        if use_mixup and labels.dtype == torch.long:
            imgs, y_a, y_b, lam = mixup_data(imgs, labels, alpha=0.4, device=device)

        ctx = torch.amp.autocast("cuda") if (scaler is not None) else torch.no_grad.__class__()
        if scaler is not None:
            with torch.amp.autocast("cuda"):
                logits = model(imgs)
                loss   = mixup_criterion(criterion, logits, y_a, y_b, lam) if (lam < 1.0)                          else criterion(logits, labels)
                loss   = loss / grad_accum
            scaler.scale(loss).backward()
            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad()
        else:
            logits = model(imgs)
            loss   = mixup_criterion(criterion, logits, y_a, y_b, lam) if (lam < 1.0)                      else criterion(logits, labels)
            (loss / grad_accum).backward()
            if (step + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); optimizer.zero_grad()

        total_loss += loss.item() * grad_accum
        if labels.dtype == torch.long:
            all_preds.extend(logits.detach().argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    train_acc = acc(all_labels, all_preds) if all_labels else 0.0
    return avg_loss, train_acc


@torch.no_grad()
def validate(model, loader, criterion=None, device=DEVICE):
    model.eval()
    total_loss = 0.0
    all_probs, all_preds, all_labels = [], [], []

    for imgs, labels in tqdm(loader, desc="  val  ", leave=False):
        imgs   = imgs.to(device)
        labels = labels.to(device)
        if USE_AMP:
            with torch.amp.autocast("cuda"):
                logits = model(imgs)
        else:
            logits = model(imgs)
        if criterion is not None:
            total_loss += criterion(logits, labels).item()
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss   = total_loss / len(loader) if criterion else 0.0
    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)
    val_acc    = acc(all_labels, all_preds)
    try:
        val_qwk = qwk(all_labels, all_preds)
    except Exception:
        val_qwk = 0.0

    return avg_loss, val_acc, val_qwk, all_probs, all_labels


# Batch size map — tune down if OOM
BS_MAP = {224: 32, 384: 16, 512: 8}

print("✅ Training engine defined: train_one_epoch, validate")
print(f"   Batch sizes: {BS_MAP}  (reduce 512→4 for 4 GB VRAM)")


## 🔁 Cell 12 — K-Fold Cross Validation Training
3-Phase Progressive Resizing: 224 → 384 → 512 px  
**Resume-safe**: completed folds are skipped automatically.

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
LR           = 2e-4
WEIGHT_DECAY = 1e-4
EPOCHS_HEAD  = 5    # Phase 1: head only,  224 px
EPOCHS_MID   = 15   # Phase 2: top-4 blocks, 384 px
EPOCHS_FULL  = 10   # Phase 3: full fine-tune, 512 px
USE_MIXUP    = True
GRAD_ACCUM   = 2    # effective batch = BS * GRAD_ACCUM

oof_probs  = np.zeros((len(df), NUM_CLASSES), dtype=np.float32)
oof_labels = df["diagnosis"].values.copy()
fold_val_qwks = []

scaler = torch.amp.GradScaler("cuda") if USE_AMP else None

print("=" * 65)
print("  5-FOLD CV  |  Progressive 224→384→512  |  Backbone:", BACKBONE)
print("=" * 65)

for fold in range(N_FOLDS):
    fold_ckpt = ARTIFACT_DIR / f"fold{fold}_best.pt"
    fold_oof  = ARTIFACT_DIR / f"fold{fold}_oof.npy"
    fold_flag = ARTIFACT_DIR / f"_done_fold{fold}.flag"

    if fold_flag.exists() and fold_ckpt.exists():
        if fold_oof.exists():
            val_idx = df[df["fold"] == fold].index
            oof_probs[val_idx] = np.load(str(fold_oof))
        prev = safe_load(fold_ckpt, "cpu")
        fold_val_qwks.append(prev.get("val_qwk", 0.0))
        print(f"  ✅ [RESUME] Fold {fold} — QWK={fold_val_qwks[-1]:.4f}")
        continue

    print(f"\n  ━━━━━━━━━  FOLD {fold}  ━━━━━━━━━")
    df_tr = df[df["fold"] != fold].reset_index(drop=True)
    df_va = df[df["fold"] == fold].reset_index(drop=True)
    val_idx = df[df["fold"] == fold].index

    criterion_fn = build_5class_criterion(df_tr)
    model = build_5class_model(pretrained=True, backbone=BACKBONE)
    best_val_qwk, best_state = -1.0, None

    # ── PHASE 1: head only, 224 px ──────────────────────────────────────────
    print(f"  Phase 1: {EPOCHS_HEAD} epochs @ 224 px (head only)")
    model.freeze_backbone()
    bs = BS_MAP[224]
    tr_ds = APTOSDataset(df_tr, build_train_transforms(224), img_size=224)
    va_ds = APTOSDataset(df_va, build_val_transforms(224),   img_size=224)
    tr_ld = make_weighted_loader(df_tr, tr_ds, bs, drop_last=True)
    va_ld = make_loader(va_ds, bs)
    opt   = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                               lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, steps_per_epoch=len(tr_ld),
        epochs=EPOCHS_HEAD, pct_start=0.3, anneal_strategy="cos")

    for ep in range(EPOCHS_HEAD):
        tl, ta = train_one_epoch(model, tr_ld, criterion_fn, opt, scaler,
                                  USE_MIXUP, GRAD_ACCUM)
        sched.step()
        vl, va_acc, vq, vp, vl_ = validate(model, va_ld, criterion_fn)
        if vq > best_val_qwk:
            best_val_qwk = vq; best_state = deepcopy(model.state_dict())
        print(f"    P1 Ep{ep+1:02d}/{EPOCHS_HEAD}: loss={tl:.4f} acc={ta*100:.1f}%"
              f" | val_loss={vl:.4f} val_acc={va_acc*100:.1f}% QWK={vq:.4f}")

    # ── PHASE 2: top-4 blocks, 384 px ───────────────────────────────────────
    print(f"  Phase 2: {EPOCHS_MID} epochs @ 384 px (top-4 blocks)")
    model.load_state_dict(best_state)  # start from P1 best
    model.unfreeze_top_blocks(n=4)
    bs = BS_MAP[384]
    tr_ds = APTOSDataset(df_tr, build_train_transforms(384), img_size=384)
    va_ds = APTOSDataset(df_va, build_val_transforms(384),   img_size=384)
    tr_ld = make_weighted_loader(df_tr, tr_ds, bs, drop_last=True)
    va_ld = make_loader(va_ds, bs)
    opt   = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                               lr=LR/3, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR/3, steps_per_epoch=len(tr_ld),
        epochs=EPOCHS_MID, pct_start=0.2, anneal_strategy="cos")

    for ep in range(EPOCHS_MID):
        tl, ta = train_one_epoch(model, tr_ld, criterion_fn, opt, scaler,
                                  USE_MIXUP, GRAD_ACCUM)
        sched.step()
        vl, va_acc, vq, vp, vl_ = validate(model, va_ld, criterion_fn)
        if vq > best_val_qwk:
            best_val_qwk = vq; best_state = deepcopy(model.state_dict())
        print(f"    P2 Ep{ep+1:02d}/{EPOCHS_MID}: loss={tl:.4f} acc={ta*100:.1f}%"
              f" | val_loss={vl:.4f} val_acc={va_acc*100:.1f}% QWK={vq:.4f}")

    # ── PHASE 3: full fine-tune, 512 px ─────────────────────────────────────
    print(f"  Phase 3: {EPOCHS_FULL} epochs @ 512 px (full fine-tune)")
    model.load_state_dict(best_state)  # start from P2 best
    model.unfreeze_all()
    bs = max(BS_MAP[512], 4)  # safety floor
    tr_ds = APTOSDataset(df_tr, build_train_transforms(512), img_size=512)
    va_ds = APTOSDataset(df_va, build_val_transforms(512),   img_size=512)
    tr_ld = make_weighted_loader(df_tr, tr_ds, bs, drop_last=True)
    va_ld = make_loader(va_ds, bs)
    opt   = torch.optim.AdamW(model.parameters(),
                               lr=LR/30, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_FULL, eta_min=1e-6)

    for ep in range(EPOCHS_FULL):
        tl, ta = train_one_epoch(model, tr_ld, criterion_fn, opt, scaler,
                                  USE_MIXUP, GRAD_ACCUM)
        sched.step()
        vl, va_acc, vq, vp, vl_ = validate(model, va_ld, criterion_fn)
        if vq > best_val_qwk:
            best_val_qwk = vq; best_state = deepcopy(model.state_dict())
        print(f"    P3 Ep{ep+1:02d}/{EPOCHS_FULL}: loss={tl:.4f} acc={ta*100:.1f}%"
              f" | val_loss={vl:.4f} val_acc={va_acc*100:.1f}% QWK={vq:.4f}  {'★' if vq>best_val_qwk-0.001 else ''}")

    # ── Save OOF & checkpoint ────────────────────────────────────────────────
    model.load_state_dict(best_state)
    model.eval()
    va_ds_final = APTOSDataset(df_va, build_val_transforms(512), img_size=512)
    va_ld_final = make_loader(va_ds_final, BS_MAP[512])
    _, _, _, fold_probs, _ = validate(model, va_ld_final, device=DEVICE)

    oof_probs[val_idx] = fold_probs
    np.save(str(fold_oof), fold_probs)
    torch.save({"model_state": best_state, "val_qwk": best_val_qwk,
                "img_size": 512, "backbone": BACKBONE}, fold_ckpt)
    fold_flag.touch()
    fold_val_qwks.append(best_val_qwk)
    print(f"  ✅ Fold {fold} complete — Best QWK: {best_val_qwk:.4f}")
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

# Save OOF arrays
np.save(str(ARTIFACT_DIR / "oof_probs.npy"),  oof_probs)
np.save(str(ARTIFACT_DIR / "oof_labels.npy"), oof_labels)
st_save("oof_qwk", float(qwk(oof_labels, oof_probs.argmax(1))))

print("\n" + "="*65)
for i, q in enumerate(fold_val_qwks):
    print(f"  Fold {i}: QWK = {q:.4f}")
print(f"  Mean QWK = {np.mean(fold_val_qwks):.4f} ± {np.std(fold_val_qwks):.4f}")
print(f"  OOF  QWK = {qwk(oof_labels, oof_probs.argmax(1)):.4f}")
print("="*65)


## 🎯 Cell 13 — Threshold Optimisation (Maximise OOF QWK)
Finds optimal ordinal thresholds [t0, t1, t2, t3] on cumulative class probabilities
to maximise QWK — more powerful than argmax alone.

In [ ]:
def optimise_thresholds(probs, labels, n_classes=5):
    """
    Optimise n_classes-1 ordinal thresholds on cumulative softmax probabilities.
    Converts P(class ≤ k) to ordinal predictions by finding optimal cut-points.

    Returns: (best_preds [N,], best_qwk float)
    """
    # Cumulative probability: P(grade ≤ k) for k in 0..n_classes-2
    cum_probs = np.cumsum(probs, axis=1)[:, :n_classes-1]  # [N, n_classes-1]

    argmax_preds = probs.argmax(1)
    baseline_qwk = qwk(labels, argmax_preds)
    print(f"  Argmax baseline QWK: {baseline_qwk:.4f}")

    def neg_qwk_from_thresholds(thresholds):
        """Map cumulative probs → ordinal class using learned thresholds."""
        # thresholds must be sorted; we enforce sorting via transformation
        t = np.sort(thresholds)
        # For each sample, find the first k where cum_probs[k] > t[k]  → grade k
        # Equivalently: grade = number of thresholds the sample exceeds
        preds = np.zeros(len(probs), dtype=int)
        for k in range(n_classes - 1):
            preds[cum_probs[:, k] > t[k]] = k + 1
        return -cohen_kappa_score(labels, preds, weights="quadratic")

    # Initialise thresholds from empirical class-frequency quantiles
    class_freq = np.bincount(labels, minlength=n_classes) / len(labels)
    t0 = np.cumsum(class_freq)[:-1]  # natural starting point
    t0 = np.clip(t0, 0.01, 0.99)

    best_qwk_val = baseline_qwk
    best_preds   = argmax_preds.copy()
    best_t       = t0.copy()

    # ── scipy L-BFGS-B optimisation ──────────────────────────────────────────
    result = minimize(
        neg_qwk_from_thresholds,
        t0,
        method="L-BFGS-B",
        bounds=[(0.01, 0.99)] * (n_classes - 1),
        options={"maxiter": 500, "ftol": 1e-9},
    )
    if -result.fun > best_qwk_val:
        best_qwk_val = -result.fun
        best_t       = np.sort(result.x)
        t_sorted = best_t
        best_preds = np.zeros(len(probs), dtype=int)
        for k in range(n_classes - 1):
            best_preds[cum_probs[:, k] > t_sorted[k]] = k + 1
        print(f"  L-BFGS-B improved → QWK={best_qwk_val:.4f}  thresholds={np.round(best_t, 3)}")

    # ── Nelder-Mead as a second pass ─────────────────────────────────────────
    result2 = minimize(
        neg_qwk_from_thresholds,
        best_t,
        method="Nelder-Mead",
        options={"maxiter": 2000, "xatol": 1e-6, "fatol": 1e-9},
    )
    if -result2.fun > best_qwk_val:
        best_qwk_val = -result2.fun
        best_t       = np.sort(result2.x)
        best_preds = np.zeros(len(probs), dtype=int)
        for k in range(n_classes - 1):
            best_preds[cum_probs[:, k] > best_t[k]] = k + 1
        print(f"  Nelder-Mead improved → QWK={best_qwk_val:.4f}")

    return best_preds, best_qwk_val, best_t


oof_probs_loaded  = np.load(str(ARTIFACT_DIR / "oof_probs.npy"))
oof_labels_loaded = np.load(str(ARTIFACT_DIR / "oof_labels.npy"))

print("Optimising thresholds on OOF predictions ...")
opt_preds, opt_qwk_val, opt_thresholds = optimise_thresholds(
    oof_probs_loaded, oof_labels_loaded)

print(f"\n  ✅ Optimised OOF QWK : {opt_qwk_val:.4f}")
print(f"     OOF Accuracy       : {acc(oof_labels_loaded, opt_preds)*100:.2f}%")
print(f"     Optimal thresholds : {np.round(opt_thresholds, 4)}")

np.save(str(ARTIFACT_DIR / "oof_opt_preds.npy"), opt_preds)
np.save(str(ARTIFACT_DIR / "opt_thresholds.npy"), opt_thresholds)
st_save("opt_qwk", float(opt_qwk_val))


## 📈 Cell 14 — OOF Confusion Matrix & Per-Class Metrics

In [ ]:
_cm_path = PLOT_DIR / "oof_confusion_matrix.png"

oof_probs_l   = np.load(str(ARTIFACT_DIR / "oof_probs.npy"))
oof_labels_l  = np.load(str(ARTIFACT_DIR / "oof_labels.npy"))
oof_opt_preds = np.load(str(ARTIFACT_DIR / "oof_opt_preds.npy"))

# Confusion matrix on optimised predictions
cm = confusion_matrix(oof_labels_l, oof_opt_preds)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=[f"G{i}" for i in range(5)])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"OOF Confusion Matrix (QWK={opt_qwk_val:.4f})", fontweight="bold")

# Per-class recall
report = classification_report(oof_labels_l, oof_opt_preds,
                                target_names=[f"G{i} {GRADE_MAP[i]}" for i in range(5)],
                                output_dict=True)
recalls   = [report[f"G{i} {GRADE_MAP[i]}"]["recall"] for i in range(5)]
precisions = [report[f"G{i} {GRADE_MAP[i]}"]["precision"] for i in range(5)]
axes[1].bar([f"G{i}" for i in range(5)], recalls,   color=GRADE_COLORS, label="Recall",    alpha=0.8)
axes[1].bar([f"G{i}" for i in range(5)], precisions, color=GRADE_COLORS, label="Precision", alpha=0.5, hatch="//")
axes[1].set_ylim(0, 1.1); axes[1].legend()
axes[1].set_title("Per-Class Recall & Precision", fontweight="bold")

plt.tight_layout()
plt.savefig(str(_cm_path), dpi=120, bbox_inches="tight")
plt.show()

# AUROC (one-vs-rest)
try:
    auroc = roc_auc_score(oof_labels_l, oof_probs_l, multi_class="ovr", average="macro")
    print(f"  AUROC (macro OvR): {auroc:.4f}")
    st_save("oof_auroc", float(auroc))
except Exception as e:
    print(f"  AUROC error: {e}")

print(f"  OOF QWK (optimised) : {opt_qwk_val:.4f}")
print(f"  OOF Accuracy        : {acc(oof_labels_l, oof_opt_preds)*100:.2f}%")
print(classification_report(oof_labels_l, oof_opt_preds,
      target_names=[f"G{i} {GRADE_MAP[i]}" for i in range(5)]))


## 🤝 Cell 15 — Ensemble Inference (5-Fold × 5 TTA views)
Tested on fold 0 held-out set using folds 1-4 models.

In [ ]:
TEST_FOLD = 0
df_test   = df[df["fold"] == TEST_FOLD].reset_index(drop=True)
print(f"Ensemble inference on fold {TEST_FOLD} ({len(df_test)} samples) ...")

ensemble_probs = np.zeros((len(df_test), NUM_CLASSES), dtype=np.float32)
n_models = 0

for fold in range(1, N_FOLDS):
    fold_ckpt = ARTIFACT_DIR / f"fold{fold}_best.pt"
    if not fold_ckpt.exists():
        print(f"  ⚠️  fold{fold}_best.pt not found — skipping")
        continue

    ckpt      = safe_load(fold_ckpt, DEVICE)
    model     = build_5class_model(pretrained=False, backbone=ckpt.get("backbone", BACKBONE))
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    img_sz    = ckpt.get("img_size", 512)
    tta_tfs   = build_tta_transforms(img_sz)

    fold_probs = np.zeros((len(df_test), NUM_CLASSES), dtype=np.float32)
    with torch.no_grad():
        for tta_tf in tta_tfs:
            ds  = APTOSDataset(df_test, transform=tta_tf, img_size=img_sz)
            ld  = make_loader(ds, batch_size=BS_MAP.get(img_sz, 8), shuffle=False)
            bp  = []
            for imgs, _ in tqdm(ld, desc=f"  fold{fold} TTA", leave=False):
                if USE_AMP:
                    with torch.amp.autocast("cuda"):
                        logits = model(imgs.to(DEVICE))
                else:
                    logits = model(imgs.to(DEVICE))
                bp.append(F.softmax(logits, 1).cpu().numpy())
            fold_probs += np.concatenate(bp, axis=0)

    fold_probs /= len(tta_tfs)                   # FIX: divide by actual TTA count
    ensemble_probs += fold_probs
    n_models += 1

    fp = fold_probs.argmax(1)
    print(f"  Fold {fold}: QWK={qwk(df_test['diagnosis'].values, fp):.4f}")
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

if n_models > 0:
    ensemble_probs /= n_models

    # Apply optimised thresholds
    opt_t      = np.load(str(ARTIFACT_DIR / "opt_thresholds.npy"))
    cum_probs  = np.cumsum(ensemble_probs, axis=1)[:, :NUM_CLASSES-1]
    test_preds = np.zeros(len(df_test), dtype=int)
    for k in range(NUM_CLASSES - 1):
        test_preds[cum_probs[:, k] > opt_t[k]] = k + 1

    test_labels = df_test["diagnosis"].values
    test_qwk_   = qwk(test_labels, test_preds)
    test_acc_   = acc(test_labels, test_preds)

    print(f"\n  ✅ Ensemble ({n_models} models × {len(tta_tfs)} TTA views)")
    print(f"     Test QWK  : {test_qwk_:.4f}")
    print(f"     Test Acc  : {test_acc_*100:.2f}%")
    st_save("test_qwk", float(test_qwk_))
    st_save("test_acc", float(test_acc_))
    np.save(str(ARTIFACT_DIR / "ensemble_test_probs.npy"), ensemble_probs)
else:
    print("⚠️  No fold models found — train Cell 12 first.")


## 🧠 Cell 16 — 2-Stage Pipeline Training
**Stage 1**: Binary (No DR vs DR) — BinaryFocalLoss + AMP  
**Stage 2**: Ordinal (grades 1–4) — per-node BCE + AMP

In [ ]:
df_tr2 = df[df["fold"] != 0].reset_index(drop=True)
df_va2 = df[df["fold"] == 0].reset_index(drop=True)

S1_CKPT = ARTIFACT_DIR / "stage1_binary_best.pt"
S2_CKPT = ARTIFACT_DIR / "stage2_ordinal_best.pt"

def _run_binary_training(df_tr, df_va, n_epochs=20, img_size=384):
    """Train Stage-1 binary classifier with AMP support."""
    print(f"  Training Stage 1 (binary): {n_epochs} epochs @ {img_size}px")
    bs = BS_MAP.get(img_size, 16)
    s1_tr_ds = BinaryAPTOSDataset(df_tr, build_train_transforms(img_size), img_size)
    s1_va_ds = BinaryAPTOSDataset(df_va, build_val_transforms(img_size),   img_size)
    # Oversample positives for binary
    bin_counts = df_tr["binary"].value_counts()
    s_wts = torch.tensor(
        [1.0 / bin_counts.get(int(df_tr.iloc[i]["binary"]), 1) for i in range(len(df_tr))],
        dtype=torch.float32)
    sampler = WeightedRandomSampler(s_wts, len(s_wts), replacement=True)
    nw = min(4, os.cpu_count() or 1)
    s1_tr_ld = DataLoader(s1_tr_ds, batch_size=bs, sampler=sampler,
                          num_workers=nw, pin_memory=(DEVICE=="cuda"), drop_last=True)
    s1_va_ld = make_loader(s1_va_ds, bs)

    model = build_stage1_model(pretrained=True)
    bfl   = BinaryFocalLoss(alpha=0.25, gamma=2.0)
    scaler_s1 = torch.amp.GradScaler("cuda") if USE_AMP else None

    def s1_loss(logits, labels):
        l = logits.squeeze(); t = labels.float().squeeze()
        return 0.5 * F.binary_cross_entropy_with_logits(l, t) + 0.5 * bfl(l.unsqueeze(1), t.unsqueeze(1))

    model.freeze_backbone()
    opt   = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                               lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, steps_per_epoch=len(s1_tr_ld), epochs=n_epochs, pct_start=0.2)

    best_loss, best_state = float("inf"), None
    for ep in range(n_epochs):
        model.train()
        ep_loss = 0.0
        for imgs, labels in tqdm(s1_tr_ld, desc=f"  S1 ep{ep+1}", leave=False):
            imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
            opt.zero_grad()
            if scaler_s1 is not None:
                with torch.amp.autocast("cuda"):
                    loss = s1_loss(model(imgs), labels)
                scaler_s1.scale(loss).backward()
                scaler_s1.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler_s1.step(opt); scaler_s1.update()
            else:
                loss = s1_loss(model(imgs), labels); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
            ep_loss += loss.item()

        # Validation
        model.eval()
        val_p, val_l = [], []
        with torch.no_grad():
            for imgs, labels in s1_va_ld:
                if USE_AMP:
                    with torch.amp.autocast("cuda"):
                        logits = model(imgs.to(DEVICE))
                else:
                    logits = model(imgs.to(DEVICE))
                val_p.extend(torch.sigmoid(logits).squeeze().cpu().numpy().tolist())
                val_l.extend(labels.cpu().numpy().tolist())
        val_p = np.array(val_p); val_l = np.array(val_l)
        val_preds_bin = (val_p >= 0.5).astype(int)
        val_ba  = acc(val_l, val_preds_bin)
        vl_mean = ep_loss / len(s1_tr_ld)
        if vl_mean < best_loss:
            best_loss = vl_mean; best_state = deepcopy(model.state_dict())
        print(f"    Ep{ep+1:02d}: loss={vl_mean:.4f}  val_bin_acc={val_ba*100:.1f}%")

        # Unfreeze top blocks at midpoint
        if ep == n_epochs // 2:
            model.unfreeze_top_blocks(n=4)
            opt   = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=LR/5, weight_decay=WEIGHT_DECAY)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=n_epochs - ep, eta_min=1e-6)
            print("    ↳ Backbone top blocks unfrozen")

    return model, best_state, best_loss


def _run_ordinal_training(df_tr, df_va, n_epochs=20, img_size=384):
    """Train Stage-2 ordinal model with AMP support."""
    print(f"  Training Stage 2 (ordinal): {n_epochs} epochs @ {img_size}px")
    bs = BS_MAP.get(img_size, 16)
    s2_tr_ds = OrdinalAPTOSDataset(df_tr, build_train_transforms(img_size), img_size)
    s2_va_ds = OrdinalAPTOSDataset(df_va, build_val_transforms(img_size),   img_size)
    s2_tr_ld = make_loader(s2_tr_ds, bs, shuffle=True, drop_last=True)
    s2_va_ld = make_loader(s2_va_ds, bs)
    crit     = OrdinalBCELoss()
    model    = build_stage2_model(pretrained=True)
    scaler_s2 = torch.amp.GradScaler("cuda") if USE_AMP else None

    model.freeze_backbone()
    opt   = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                               lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, steps_per_epoch=len(s2_tr_ld), epochs=n_epochs, pct_start=0.2)

    best_loss, best_state = float("inf"), None
    for ep in range(n_epochs):
        model.train()
        ep_loss = 0.0
        for imgs, labels in tqdm(s2_tr_ld, desc=f"  S2 ep{ep+1}", leave=False):
            imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
            opt.zero_grad()
            if scaler_s2 is not None:
                with torch.amp.autocast("cuda"):
                    loss = crit(model(imgs), labels)
                scaler_s2.scale(loss).backward()
                scaler_s2.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler_s2.step(opt); scaler_s2.update()
            else:
                loss = crit(model(imgs), labels); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step(); ep_loss += loss.item()

        vl = ep_loss / len(s2_tr_ld)
        if vl < best_loss:
            best_loss = vl; best_state = deepcopy(model.state_dict())
        print(f"    Ep{ep+1:02d}: loss={vl:.4f}")
        if ep == n_epochs // 2:
            model.unfreeze_top_blocks(n=4)
            opt   = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=LR/5, weight_decay=WEIGHT_DECAY)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=n_epochs - ep, eta_min=1e-6)

    return model, best_state, best_loss


# ── Stage 1 ──────────────────────────────────────────────────────────────────
if (ARTIFACT_DIR / "_done_stage1.flag").exists() and S1_CKPT.exists():
    print("✅ [RESUME] Stage 1 already trained.")
else:
    s1_model, s1_state, s1_loss = _run_binary_training(df_tr2, df_va2, n_epochs=20)
    torch.save({"model_state": s1_state, "val_loss": s1_loss,
                "img_size": 384, "backbone": BACKBONE}, S1_CKPT)
    (ARTIFACT_DIR / "_done_stage1.flag").touch()
    del s1_model; gc.collect()
    print(f"  ✅ Stage 1 saved → {S1_CKPT}")

# ── Stage 2 ──────────────────────────────────────────────────────────────────
if (ARTIFACT_DIR / "_done_stage2.flag").exists() and S2_CKPT.exists():
    print("✅ [RESUME] Stage 2 already trained.")
else:
    s2_model, s2_state, s2_loss = _run_ordinal_training(df_tr2, df_va2, n_epochs=20)
    torch.save({"model_state": s2_state, "val_loss": s2_loss,
                "img_size": 384, "backbone": BACKBONE}, S2_CKPT)
    (ARTIFACT_DIR / "_done_stage2.flag").touch()
    del s2_model; gc.collect()
    print(f"  ✅ Stage 2 saved → {S2_CKPT}")


## 🔮 Cell 17 — 2-Stage Combined Inference
```
Stage1_prob < threshold  →  Grade 0
else  →  Grade 1-4  =  count(sigmoid nodes ≥ 0.5) + 1
```

In [ ]:
@torch.no_grad()
def two_stage_predict(image_paths, stage1_threshold=0.5, tta=True, device=DEVICE):
    """
    2-stage DR grading. Returns (final_grades, s1_probs).
    """
    if not S1_CKPT.exists() or not S2_CKPT.exists():
        raise FileNotFoundError("Stage 1 or Stage 2 checkpoint missing — run Cell 16 first.")

    s1_ckpt = safe_load(S1_CKPT, device)
    s2_ckpt = safe_load(S2_CKPT, device)
    img_sz  = s1_ckpt.get("img_size", 384)

    s1_model = build_stage1_model(pretrained=False,
                                   backbone=s1_ckpt.get("backbone", BACKBONE))
    s1_model.load_state_dict(s1_ckpt["model_state"]); s1_model.eval()

    s2_model = build_stage2_model(pretrained=False,
                                   backbone=s2_ckpt.get("backbone", BACKBONE))
    s2_model.load_state_dict(s2_ckpt["model_state"]); s2_model.eval()

    df_inf   = pd.DataFrame({"image_path": image_paths, "diagnosis": [0]*len(image_paths)})
    tta_tfs  = build_tta_transforms(img_sz) if tta else [build_val_transforms(img_sz)]

    # ── Stage 1 ──────────────────────────────────────────────────────────────
    s1_probs = np.zeros(len(df_inf), dtype=np.float32)
    for tf in tta_tfs:
        ds = APTOSDataset(df_inf, tf, img_size=img_sz)
        ld = make_loader(ds, 8)
        bp = []
        for imgs, _ in ld:
            if USE_AMP:
                with torch.amp.autocast("cuda"):
                    logits = s1_model(imgs.to(device))
            else:
                logits = s1_model(imgs.to(device))
            p = torch.sigmoid(logits).squeeze()
            bp.extend((p.cpu().numpy() if p.dim() > 0 else [float(p.cpu())]))
        s1_probs += np.array(bp[:len(df_inf)])
    s1_probs /= len(tta_tfs)
    s1_preds = (s1_probs >= stage1_threshold).astype(int)

    # ── Stage 2 (DR-positive only) ───────────────────────────────────────────
    dr_idx = np.where(s1_preds == 1)[0]
    final_grades = np.zeros(len(df_inf), dtype=int)

    if len(dr_idx) > 0:
        df_dr = df_inf.iloc[dr_idx].reset_index(drop=True)
        s2_node = np.zeros((len(df_dr), 4), dtype=np.float32)
        for tf in tta_tfs:
            ds = APTOSDataset(df_dr, tf, img_size=img_sz)
            ld = make_loader(ds, 8)
            bp = []
            for imgs, _ in ld:
                if USE_AMP:
                    with torch.amp.autocast("cuda"):
                        logits = s2_model(imgs.to(device))
                else:
                    logits = s2_model(imgs.to(device))
                bp.append(torch.sigmoid(logits).cpu().numpy())
            s2_node += np.concatenate(bp, axis=0)[:len(df_dr)]
        s2_node /= len(tta_tfs)
        s2_grades = np.clip((s2_node >= 0.5).sum(axis=1) + 1, 1, 4)
        final_grades[dr_idx] = s2_grades

    del s1_model, s2_model; gc.collect()
    return final_grades, s1_probs


# ── Evaluate on validation fold 0 ────────────────────────────────────────────
print("Evaluating 2-stage pipeline on val fold 0 ...")
val_paths  = df_va2["image_path"].tolist()
val_grades = df_va2["diagnosis"].values

pred_grades, s1_probs_out = two_stage_predict(val_paths, stage1_threshold=0.5, tta=True)

ts_qwk  = qwk(val_grades, pred_grades)
ts_acc_ = acc(val_grades, pred_grades)
try:
    ts_auroc = roc_auc_score((val_grades >= 1).astype(int), s1_probs_out)
except Exception:
    ts_auroc = float("nan")

print(f"  2-Stage QWK   : {ts_qwk:.4f}")
print(f"  2-Stage Acc   : {ts_acc_*100:.2f}%")
print(f"  Stage-1 AUROC : {ts_auroc:.4f}")
st_save("two_stage_qwk",  float(ts_qwk))
st_save("two_stage_acc",  float(ts_acc_))
st_save("stage1_auroc",   float(ts_auroc))


## 🎨 Cell 18 — Grad-CAM++ Explainability

In [ ]:
try:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

    # Load best fold model
    _best_fold = int(np.argmax(fold_val_qwks)) if fold_val_qwks else 0
    _ckpt = safe_load(ARTIFACT_DIR / f"fold{_best_fold}_best.pt", DEVICE)
    _model = build_5class_model(pretrained=False, backbone=_ckpt.get("backbone", BACKBONE))
    _model.load_state_dict(_ckpt["model_state"]); _model.eval()

    # Target layer: last conv block of EfficientNetV2
    if hasattr(_model.backbone, "blocks"):
        _target_layer = [_model.backbone.blocks[-1][-1]]
    else:
        _target_layer = [list(_model.backbone.children())[-1]]

    cam = GradCAMPlusPlus(model=_model, target_layers=_target_layer)

    # Show Grad-CAM on a sample from each grade
    n_cols = 5
    fig, axes = plt.subplots(2, n_cols, figsize=(20, 8))
    for grade in range(5):
        samples = df[df["diagnosis"] == grade]
        if len(samples) == 0: continue
        sample  = samples.sample(1, random_state=SEED).iloc[0]
        raw_img = preprocess_fundus(sample["image_path"], size=512)
        tf      = build_val_transforms(512)
        tensor  = tf(image=raw_img)["image"].unsqueeze(0).to(DEVICE)

        grayscale_cam = cam(input_tensor=tensor,
                            targets=[ClassifierOutputTarget(grade)])[0]
        vis = show_cam_on_image(raw_img.astype(np.float32)/255.0,
                                grayscale_cam, use_rgb=True)

        axes[0][grade].imshow(raw_img)
        axes[0][grade].set_title(f"Grade {grade}: {GRADE_MAP[grade]}", fontsize=10)
        axes[0][grade].axis("off")
        axes[1][grade].imshow(vis)
        axes[1][grade].set_title("Grad-CAM++", fontsize=10)
        axes[1][grade].axis("off")

    plt.suptitle("Grad-CAM++ Attention Maps — All 5 DR Grades", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(PLOT_DIR / "gradcam_all_grades.png"), dpi=120, bbox_inches="tight")
    plt.show()
    del _model, cam; gc.collect()
    print("✅ Grad-CAM++ visualisation complete.")

except ImportError:
    print("⚠️  grad-cam not installed — run: pip install grad-cam")
except Exception as e:
    print(f"⚠️  Grad-CAM error: {e}")


## 📋 Cell 19 — Final Results Summary

In [ ]:
state = st_load()

print("=" * 65)
print("  DIABETIC RETINOPATHY GRADING — v17 FINAL SUMMARY")
print("=" * 65)
print(f"  Backbone     : {BACKBONE}")
print(f"  Device       : {DEVICE.upper()}  (AMP: {'ON' if USE_AMP else 'OFF'})")
print(f"  Dataset      : APTOS 2019 ({len(df):,} images, 5 classes)")
print(f"  Strategy     : 5-Fold CV | 3-phase progressive resize | 5×TTA ensemble")
print()
print("  ─── K-Fold Cross Validation ───")
if "fold_val_qwks" in dir() and fold_val_qwks:
    for i, q in enumerate(fold_val_qwks):
        print(f"    Fold {i}: QWK = {q:.4f}")
    print(f"    Mean : {np.mean(fold_val_qwks):.4f} ± {np.std(fold_val_qwks):.4f}")
print(f"    OOF QWK (argmax)   : {state.get('oof_qwk', 'N/A')}")
print(f"    OOF QWK (optimised): {state.get('opt_qwk', 'N/A')}")
print(f"    OOF AUROC (macro)  : {state.get('oof_auroc', 'N/A')}")
print()
print("  ─── Ensemble (fold 0 test) ───")
tqwk = state.get("test_qwk", "N/A")
tacc = float(state.get("test_acc", 0)) * 100
print(f"    Test QWK  : {tqwk}")
print(f"    Test Acc  : {tacc:.2f}%")
print()
print("  ─── 2-Stage Pipeline (val fold 0) ───")
tsqwk = state.get("two_stage_qwk", "N/A")
tsacc = float(state.get("two_stage_acc", 0)) * 100
print(f"    2-Stage QWK   : {tsqwk}")
print(f"    2-Stage Acc   : {tsacc:.2f}%")
print(f"    Stage-1 AUROC : {state.get('stage1_auroc', 'N/A')}")
print()
print("  Artifacts:")
for f in sorted(ARTIFACT_DIR.glob("*.pt")):
    print(f"    {f.name:<35s} {f.stat().st_size/1e6:.1f} MB")
print("=" * 65)
print("  ⚠️  RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT")
print("=" * 65)
